In [21]:
import ROOT
import numpy as np
import pandas as pd
 
 # ROOT.gStyle.SetOptFit(0000)    # aggiunge il label con chi2/ndf e probabilita e parametri

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    dx=10e-6
    return (model(p + dx, par) - model(p, par)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
print(p)
q = df[:, 6]

err_p = df[:, 7]

err_q = df[:,8]

p = np.delete(p, [0,1, 10, 12, 14, 43, 44, 52] )
q = np.delete(q, [0,1, 10, 12, 14, 43, 44, 52] )
err_p = np.delete(err_p, [0,1, 10, 12, 14, 43, 44, 52])
err_q = np.delete(err_q, [0,1, 10, 12, 14, 43, 44, 52])




#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParLimits(0, 0, 50) 
model_fit.SetParLimits(1, 0, 50)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("APX")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

[ 232.5  233.5  234.5  235.5  236.5  237.5  238.5  239.5  240.5  241.5
  242.5  243.5  244.5  245.5  246.5  247.5  248.5  249.5  250.5  251.5
  252.5  253.5  254.5  255.5  256.5  257.5  258.5  259.5  260.5  261.5
  262.5  263.5  264.5  265.5  267.5  269.5  271.5  273.5  275.5  277.5
  280.5  281.5  282.5  283.5  284.5  285.5  286.5  287.5  288.5  289.5
  290.5  291.5  292.5  293.5  294.5  295.5  297.5  299.5  301.5  306.5
  311.5  316.5  321.5  326.5  331.5  336.5  341.5  346.5  351.5  356.5
  361.5  366.5  371.5  376.5  381.5  386.5  391.5  396.5  401.5  406.5
  411.5  416.5  421.5  426.5  431.5  441.5  451.5  461.5  471.5  481.5
  491.5  501.5  511.5  521.5  531.5  541.5  551.5  561.5  571.5  581.5
  601.5  621.5  641.5  661.5  681.5  731.5  781.5  831.5  881.5  981.5
 1081.5 1181.5 1281.5 1381.5 1481.5 1631.5]
Fit status: 0
Parametri stimati:
A = 1.8247836010073266 ± 5.519174746000009
B = 9.087617183767271e-06 ± 44.39774030966309
C = 201.66952242203652 ± 2.8296436184939857
Chi2 = 0.

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [22]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [23]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff

#residuals_norm = np.delete(residuals_norm, 0)

print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


[ 6.23382494e-04 -1.29791079e-02 -1.01549714e-02 -9.07448395e-03
 -6.99968126e-03 -1.18786997e-02 -7.08405496e-03 -1.18498283e-02
 -1.89308662e-03 -1.13339709e-02  1.57679454e-04  1.81058230e-03
  4.67807012e-04 -3.51725526e-03 -1.77812914e-03 -8.19991998e-03
 -3.78782621e-04 -2.51253257e-03  3.87955044e-03 -2.92467572e-03
 -5.08950336e-03 -1.12199814e-02 -1.24704223e-02 -3.89666108e-03
 -9.89153834e-04 -6.99075905e-03  4.80061222e-03 -2.37338117e-03
 -2.53753872e-03 -3.44568381e-03 -1.27316313e-03 -8.62460830e-04
  3.73822904e-03  2.35818821e-03  1.14182965e-02  1.42791588e-04
  5.75199409e-03  3.64512471e-03 -6.54605177e-04 -1.10267754e-02
 -7.30841125e-03  3.83942164e-03 -5.24466694e-03 -3.14978289e-03
  7.10372357e-03  2.20307869e-03 -4.02515327e-03  8.64817953e-03
  3.36323007e-03  7.44019761e-03 -8.62665165e-03 -3.12562448e-03
  2.36191976e-03 -1.16153346e-03 -1.62815993e-03 -8.76192010e-04
 -7.00783977e-04 -8.08344532e-04 -7.09315315e-04 -6.84096803e-04
 -8.24705478e-04 -9.29692

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [24]:
if np.abs(parameters[0] - 1 )< 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1] )< 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")
#print(3*errors[2])

if np.abs(parameters[2] - 1 )< 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [25]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[  1.24093511 -24.09324496 -17.36255384 -15.07188981 -11.3693797
 -15.08721606  -8.47841877 -15.19526302  -1.8033165   -9.82759633
   0.14057395   1.32671118   0.38277926  -2.79205592  -1.12920688
  -4.8068185   -0.23129536  -1.93312135   2.51110078  -1.67873618
  -2.98348264  -5.20694046  -6.54872103  -1.76076435  -0.49528431
  -3.07350351   1.934717    -0.95650736  -1.27058408  -1.17825785
  -0.41981186  -0.32651862   1.27829422   0.89278532   3.20727393
   0.05580325   2.31813782   1.4245223   -0.1838714   -2.69330437
  -1.3388169    0.84400543  -1.08886432  -0.65393785   1.64833914
   0.43048392  -0.5898895    1.26739778   0.65717873   1.18123091
  -1.68565691  -0.53440658   0.25960602 -12.5681777  -17.3189709
  -9.30947962  -7.39443116  -8.45039741  -7.37183764  -7.06797148
  -8.46028571  -9.48052989  -8.06909257  -6.17367631  -7.74820939
  -8.75194608 -10.14871869 -10.90631218 -13.99593851 -15.39179279
  -7.07067658  -4.01167698  -2.19589185  -2.60619388  -1.22702701
  -1.0442303

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created
